In [3]:
#ejercicio Predecir futuro de una app


from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

class App:
    def __init__(self, app_name, monthly_users, avg_session_length, retention_rate, social_shares, success=None):
        self.app_name = app_name
        self.monthly_users = monthly_users
        self.avg_session_length = avg_session_length
        self.retention_rate = retention_rate
        self.social_shares = social_shares
        self.success = success

    def to_features(self):
        # Devuelve una lista con las 4 métricas numéricas de la app.
        # El orden importa: debe ser siempre el mismo al entrenar y al predecir,
        # de lo contrario el modelo recibiría columnas en posiciones incorrectas.
        return [self.monthly_users, self.avg_session_length,
                self.retention_rate, self.social_shares]


class AppDataset:
    def __init__(self, apps):
        self.apps = apps

    def get_feature_matrix(self):
        # Llama a to_features() en cada app y agrupa el resultado en una matriz.
        # Cada fila = una app, cada columna = una métrica.
        # Ejemplo con 6 apps:
        #   [[10000, 12.5, 0.65, 1500],
        #    [  500,  5.0, 0.20,   50], ...]
        return [app.to_features() for app in self.apps]

    def get_target_vector(self):
        # Recorre todas las apps y recoge solo las etiquetas definidas (0 o 1).
        # Se omiten las apps con success=None porque aún no se sabe si tuvieron éxito;
        # incluirlas rompería el entrenamiento al no tener etiqueta válida.
        # Resultado: [1, 0, 1, 0, 1, 0]
        return [app.success for app in self.apps if app.success is not None]


class SuccessPredictor:
    def __init__(self):
        self.model = LogisticRegression()
        self.scaler = StandardScaler()

    def train(self, dataset: AppDataset):
        X = dataset.get_feature_matrix()   # matriz de features
        y = dataset.get_target_vector()    # vector de etiquetas

        # fit_transform hace dos cosas en un solo paso:
        #   1. fit   → calcula la media y desviación estándar de cada columna
        #              y las guarda dentro del scaler para usarlas después.
        #   2. transform → aplica la fórmula z = (x - media) / std a cada valor,
        #              dejando todas las columnas en la misma escala (~0 a 1).
        # Esto evita que monthly_users (miles) domine sobre retention_rate (0-1).
        X_scaled = self.scaler.fit_transform(X)

        # Entrena la regresión logística con los datos ya escalados.
        self.model.fit(X_scaled, y)

    def predict(self, app: App):
        X = [app.to_features()]   # debe ser una lista de listas, no una lista plana

        # IMPORTANTE: aquí se usa transform (sin fit).
        # Si volviéramos a hacer fit_transform, el scaler recalcularía la media
        # y la std solo con esta app, obteniendo una escala completamente distinta
        # a la del entrenamiento → el modelo recibiría datos en otro "idioma".
        X_scaled = self.scaler.transform(X)

        # predict devuelve un array, tomamos el primer (y único) elemento.
        # Lo convertimos a int para devolver 0 o 1 limpiamente.
        return int(self.model.predict(X_scaled)[0])

    def predict_proba(self, app: App):
        X = [app.to_features()]
        X_scaled = self.scaler.transform(X)   # misma razón que en predict

        # predict_proba devuelve [[prob_clase_0, prob_clase_1]].
        # [0] accede a la primera (y única) fila.
        # [1] accede a la probabilidad de la clase 1 (éxito).
        return float(self.model.predict_proba(X_scaled)[0][1])


# Datos de entrenamiento
apps = [
    App("FastChat",    10000, 12.5, 0.65, 1500, 1),
    App("FitTrack",      500,  5.0, 0.20,   50, 0),
    App("GameHub",     15000, 25.0, 0.75, 3000, 1),
    App("BudgetBuddy",   800,  6.5, 0.30,   80, 0),
    App("EduFlash",    12000, 18.0, 0.70, 2200, 1),
    App("NoteKeeper",    600,  4.0, 0.15,   30, 0),
]

dataset = AppDataset(apps)
predictor = SuccessPredictor()
predictor.train(dataset)

# Nueva app a evaluar (sin etiqueta: el modelo la predecirá)
new_app = App("StudyBoost", 4000, 15.0, 0.5, 700)
predicted_success = predictor.predict(new_app)
prob = predictor.predict_proba(new_app)

print(f"¿Será exitosa la app {new_app.app_name}? {'Sí' if predicted_success else 'No'}")
print(f"Probabilidad estimada de éxito: {prob:.2f}")

¿Será exitosa la app StudyBoost? No
Probabilidad estimada de éxito: 0.49
